In [ ]:
# Celula 1: Clonar ou atualizar o repositorio no Google Colab
import os
import subprocess

REPO_URL = "https://github.com/lucasrs012007-dev/gps-canal-dark.git"
REPO_DIR = "/content/gps-canal-dark"

if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run(["git", "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

!ls -la

In [ ]:
%%bash
# Celula 2: Gerar audio com edge-tts (se ainda nao existir), garantir ffmpeg + Chrome e renderizar com legendas
set -e

if ! command -v ffmpeg >/dev/null 2>&1; then
  apt-get update -qq && apt-get install -y -qq ffmpeg
fi

python3 -m pip install -q "edge-tts>=7"

if [ ! -s audio/narracao.mp3 ]; then
  echo "Gerando audio/narracao.mp3 e audio/timestamps.json..."
  python3 audio/gerar_narracao.py
else
  echo "Usando audio/narracao.mp3 ja existente."
fi

DURACAO=$(ffprobe -v error -show_entries format=duration -of default=nw=1:nk=1 audio/narracao.mp3)
echo "Duracao real de audio/narracao.mp3: ${DURACAO}s"

if ! command -v google-chrome >/dev/null 2>&1 && ! command -v chromium >/dev/null 2>&1; then
  wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
  apt-get update -qq && apt-get install -y -qq /tmp/chrome.deb
  rm -f /tmp/chrome.deb
fi

# Troque por --teste se quiser validar apenas os 2 primeiros segundos antes do render completo.
python3 render/renderizar.py --legendas

In [ ]:
# Celula 3: Conferir audio, metadados do MP4 e baixar output/short_01.mp4
from IPython.display import Audio, display
from google.colab import files

print("Previa da narracao (audio/narracao.mp3):")
display(Audio("audio/narracao.mp3"))

!ffprobe -v error -show_entries format=duration:stream=codec_type,codec_name,width,height,r_frame_rate,pix_fmt -of default=nw=1 output/short_01.mp4

files.download("output/short_01.mp4")